# 04 - Exercise 3 Walkthrough

This notebook is the learning path for Exercise 3.

Recommended order:

1. run `00_setup_and_load_data.ipynb`
2. run `01_data_eda_and_assumptions.ipynb`
3. understand the forecasting logic in this walkthrough
4. run `05_exercise_3_production_solution.ipynb` to produce the final files quickly

Target question:

> Select the top 1 user who has the highest number of sessions. Forecast the next 3 months of your selected metric, starting from the last available record for that user.


## Functions used in this notebook

### `create_spark_session(...)`
- **Description:** Starts the Spark environment for the walkthrough.
- **Input:** Notebook app name and optional Spark settings.
- **Output:** Configured `SparkSession`.
- **Why this operation is selected for efficiency:** The explanatory notebook shares the same environment as the production notebook, so what is learned here transfers directly.

### `load_lastfm_events(...)`
- **Description:** Loads the normalized events table.
- **Input:** Spark session and project root.
- **Output:** Spark DataFrame of play events.
- **Why this operation is selected for efficiency:** It reuses Parquet and keeps the walkthrough fast enough to rerun while studying.

### `summarize_sessions_from_events(...)`
- **Description:** Builds the session summary table used as the base for the ML problem.
- **Input:** Event DataFrame and the 20-minute session gap.
- **Output:** Session-level Spark DataFrame.
- **Why this operation is selected for efficiency:** The forecasting problem depends on session-level metrics, so summarizing once keeps the rest of the pipeline smaller.

### `top_users_by_session_count(...)`
- **Description:** Ranks users by total number of sessions.
- **Input:** Session summary DataFrame and the number of users to keep.
- **Output:** Ranked user table.
- **Why this operation is selected for efficiency:** The ML exercise only needs the single top user, so identifying that user early trims the downstream data.

### `get_top_user_for_ml(session_summary_df)`
- **Description:** Returns the single user selected for forecasting.
- **Input:** Session summary DataFrame.
- **Output:** Python dictionary with `user_id`, total sessions, and date coverage.
- **Why this operation is selected for efficiency:** It packages the selection logic in one small step so the downstream forecast code stays readable.

### `build_monthly_metric_history(session_summary_df, user_id, metric='session_count')`
- **Description:** Builds a month-by-month history for the selected user and fills missing months with zero sessions.
- **Input:** Session summary DataFrame, user ID, and selected metric name.
- **Output:** Pandas DataFrame with a continuous monthly series and the selected metric.
- **Why this operation is selected for efficiency:** The aggregation happens in Spark first, and only the small per-user monthly series is converted to Pandas for lightweight forecasting.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


WindowsPath('C:/Users/GonzaloFigueroa/Documents/Private/BME/coding challenge')

In [2]:
import numpy as np
import pandas as pd
from pyspark import StorageLevel
from pyspark.sql import functions as F

from src.spark_utils import create_spark_session
from src.sessionization import (
    ensure_lastfm_dataset_available,
    load_lastfm_events,
    resolve_lastfm_input_path,
    stage_lastfm_events_to_parquet,
    summarize_sessions_from_events,
    top_users_by_session_count,
)
from src.forecasting import build_monthly_metric_history, get_top_user_for_ml


In [3]:
spark = create_spark_session(app_name='04-exercise-3-walkthrough')
spark


In [4]:
raw_input_path = ensure_lastfm_dataset_available(PROJECT_ROOT)
parquet_path = stage_lastfm_events_to_parquet(spark, PROJECT_ROOT)
print(f'Raw dataset path: {raw_input_path}')
print(f'Prepared Parquet path: {parquet_path}')


Raw dataset path: C:\Users\GonzaloFigueroa\Documents\Private\BME\coding challenge\data\raw\lastfm-dataset-1K\lastfm-dataset-1K\userid-timestamp-artid-artname-traid-traname.tsv
Prepared Parquet path: C:\Users\GonzaloFigueroa\Documents\Private\BME\coding challenge\data\processed\lastfm_events_parquet


In [5]:
events_df = load_lastfm_events(spark, PROJECT_ROOT)
session_summary_df = summarize_sessions_from_events(events_df, gap_minutes=20).persist(StorageLevel.DISK_ONLY)
session_summary_df.count()


1041883

In [6]:
top_users_df = top_users_by_session_count(
    session_summary_df=session_summary_df,
    top_user_count=10,
)

top_users_df.show(10, truncate=False)


+-----------+-------------+-------------------+-------------------+
|user_id    |session_count|first_session_start|last_session_end   |
+-----------+-------------+-------------------+-------------------+
|user_000833|6897         |2005-02-20 02:42:32|2009-05-23 14:06:37|
|user_000233|6493         |2005-07-20 01:35:09|2009-06-18 06:25:34|
|user_000861|6056         |2005-09-10 05:48:52|2009-04-29 04:11:56|
|user_000783|5575         |2006-02-13 15:55:27|2009-04-28 17:50:40|
|user_000384|5543         |2005-02-14 08:01:11|2009-05-01 18:46:31|
|user_000625|5499         |2006-03-04 23:03:49|2009-06-15 21:33:37|
|user_000792|5334         |2005-02-14 00:06:00|2009-05-22 20:28:22|
|user_000089|5302         |2005-12-25 16:22:54|2009-04-30 00:37:57|
|user_000348|5076         |2005-02-14 19:39:44|2009-06-18 14:16:53|
|user_000751|4871         |2005-07-09 11:57:48|2009-06-19 20:22:22|
+-----------+-------------+-------------------+-------------------+



In [7]:
top_user_info = get_top_user_for_ml(session_summary_df)
top_user_info


{'user_id': 'user_000833',
 'session_count': 6897,
 'first_session_start': datetime.datetime(2005, 2, 20, 3, 42, 32),
 'last_session_end': datetime.datetime(2009, 5, 23, 16, 6, 37)}

In [8]:
metric_name = 'session_count'
metric_name


'session_count'

In [9]:
history_df = build_monthly_metric_history(
    session_summary_df=session_summary_df,
    user_id=top_user_info['user_id'],
    metric=metric_name,
)

history_df.head(12)


,month_start,session_count,avg_session_duration_minutes,user_id,selected_metric,metric_name
0,2005-02-01,17.0,110.118627,user_000833,17.0,session_count
1,2005-03-01,44.0,95.695076,user_000833,44.0,session_count
2,2005-04-01,63.0,80.227513,user_000833,63.0,session_count
3,2005-05-01,33.0,86.400505,user_000833,33.0,session_count
4,2005-06-01,89.0,58.637079,user_000833,89.0,session_count
5,2005-07-01,51.0,44.660458,user_000833,51.0,session_count
6,2005-08-01,76.0,57.679605,user_000833,76.0,session_count
7,2005-09-01,65.0,43.321026,user_000833,65.0,session_count
8,2005-10-01,116.0,66.280747,user_000833,116.0,session_count
9,2005-11-01,112.0,58.840774,user_000833,112.0,session_count


In [10]:
history_df.tail(12)


,month_start,session_count,avg_session_duration_minutes,user_id,selected_metric,metric_name
40,2008-06-01,195.0,81.435470,user_000833,195.0,session_count
41,2008-07-01,237.0,65.645077,user_000833,237.0,session_count
42,2008-08-01,192.0,56.794010,user_000833,192.0,session_count
43,2008-09-01,143.0,60.128322,user_000833,143.0,session_count
44,2008-10-01,229.0,63.666448,user_000833,229.0,session_count
45,2008-11-01,231.0,53.327561,user_000833,231.0,session_count
46,2008-12-01,227.0,49.967401,user_000833,227.0,session_count
47,2009-01-01,219.0,52.595282,user_000833,219.0,session_count
48,2009-02-01,197.0,41.222081,user_000833,197.0,session_count
49,2009-03-01,192.0,38.011719,user_000833,192.0,session_count


In [11]:
model_history_df = history_df[['month_start', 'selected_metric']].copy()
model_history_df.head()


,month_start,selected_metric
0,2005-02-01,17.0
1,2005-03-01,44.0
2,2005-04-01,63.0
3,2005-05-01,33.0
4,2005-06-01,89.0


In [12]:
def add_time_series_features(history_frame: pd.DataFrame, target_col: str = 'selected_metric') -> pd.DataFrame:
    feature_df = history_frame.copy().sort_values('month_start').reset_index(drop=True)
    feature_df['time_index'] = np.arange(len(feature_df), dtype=float)
    feature_df['month_number'] = feature_df['month_start'].dt.month.astype(float)
    feature_df['month_sin'] = np.sin(2.0 * np.pi * feature_df['month_number'] / 12.0)
    feature_df['month_cos'] = np.cos(2.0 * np.pi * feature_df['month_number'] / 12.0)
    feature_df['lag_1'] = feature_df[target_col].shift(1)
    feature_df['lag_2'] = feature_df[target_col].shift(2)
    feature_df['lag_3'] = feature_df[target_col].shift(3)
    feature_df['rolling_mean_3'] = feature_df[['lag_1', 'lag_2', 'lag_3']].mean(axis=1)
    return feature_df

feature_df = add_time_series_features(model_history_df)
feature_df.head(12)


,month_start,selected_metric,time_index,month_number,month_sin,month_cos,lag_1,lag_2,lag_3,rolling_mean_3
0,2005-02-01,17.0,0.0,2.0,8.660254e-01,5.000000e-01,NaN,NaN,NaN,NaN
1,2005-03-01,44.0,1.0,3.0,1.000000e+00,6.123234e-17,17.0,NaN,NaN,17.000000
2,2005-04-01,63.0,2.0,4.0,8.660254e-01,-5.000000e-01,44.0,17.0,NaN,30.500000
3,2005-05-01,33.0,3.0,5.0,5.000000e-01,-8.660254e-01,63.0,44.0,17.0,41.333333
4,2005-06-01,89.0,4.0,6.0,1.224647e-16,-1.000000e+00,33.0,63.0,44.0,46.666667
5,2005-07-01,51.0,5.0,7.0,-5.000000e-01,-8.660254e-01,89.0,33.0,63.0,61.666667
6,2005-08-01,76.0,6.0,8.0,-8.660254e-01,-5.000000e-01,51.0,89.0,33.0,57.666667
7,2005-09-01,65.0,7.0,9.0,-1.000000e+00,-1.836970e-16,76.0,51.0,89.0,72.000000
8,2005-10-01,116.0,8.0,10.0,-8.660254e-01,5.000000e-01,65.0,76.0,51.0,64.000000
9,2005-11-01,112.0,9.0,11.0,-5.000000e-01,8.660254e-01,116.0,65.0,76.0,85.666667


In [13]:
feature_cols = [
    'time_index',
    'month_sin',
    'month_cos',
    'lag_1',
    'lag_2',
    'lag_3',
    'rolling_mean_3',
]

feature_cols


['time_index',
 'month_sin',
 'month_cos',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_mean_3']

In [14]:
trainable_df = feature_df.dropna(subset=feature_cols + ['selected_metric']).reset_index(drop=True)
print(f'Rows available for modeling: {len(trainable_df)}')
trainable_df.head(10)


Rows available for modeling: 49


,month_start,selected_metric,time_index,month_number,month_sin,month_cos,lag_1,lag_2,lag_3,rolling_mean_3
0,2005-05-01,33.0,3.0,5.0,5.000000e-01,-8.660254e-01,63.0,44.0,17.0,41.333333
1,2005-06-01,89.0,4.0,6.0,1.224647e-16,-1.000000e+00,33.0,63.0,44.0,46.666667
2,2005-07-01,51.0,5.0,7.0,-5.000000e-01,-8.660254e-01,89.0,33.0,63.0,61.666667
3,2005-08-01,76.0,6.0,8.0,-8.660254e-01,-5.000000e-01,51.0,89.0,33.0,57.666667
4,2005-09-01,65.0,7.0,9.0,-1.000000e+00,-1.836970e-16,76.0,51.0,89.0,72.000000
5,2005-10-01,116.0,8.0,10.0,-8.660254e-01,5.000000e-01,65.0,76.0,51.0,64.000000
6,2005-11-01,112.0,9.0,11.0,-5.000000e-01,8.660254e-01,116.0,65.0,76.0,85.666667
7,2005-12-01,121.0,10.0,12.0,-2.449294e-16,1.000000e+00,112.0,116.0,65.0,97.666667
8,2006-01-01,123.0,11.0,1.0,5.000000e-01,8.660254e-01,121.0,112.0,116.0,116.333333
9,2006-02-01,97.0,12.0,2.0,8.660254e-01,5.000000e-01,123.0,121.0,112.0,118.666667


In [15]:
holdout_months = 6
model_train_df = trainable_df.iloc[:-holdout_months].copy()
validation_df = trainable_df.iloc[-holdout_months:].copy()

print(f'Training rows: {len(model_train_df)}')
print(f'Validation rows: {len(validation_df)}')


Training rows: 43
Validation rows: 6


In [16]:
model_train_df[['month_start', 'selected_metric']].tail(8)


,month_start,selected_metric
35,2008-04-01,153.0
36,2008-05-01,143.0
37,2008-06-01,195.0
38,2008-07-01,237.0
39,2008-08-01,192.0
40,2008-09-01,143.0
41,2008-10-01,229.0
42,2008-11-01,231.0


In [17]:
validation_df[['month_start', 'selected_metric']]


,month_start,selected_metric
43,2008-12-01,227.0
44,2009-01-01,219.0
45,2009-02-01,197.0
46,2009-03-01,192.0
47,2009-04-01,153.0
48,2009-05-01,128.0


In [18]:
x_train = model_train_df[feature_cols].to_numpy(dtype=float)
x_train = np.column_stack([np.ones(len(x_train)), x_train])
y_train = model_train_df['selected_metric'].to_numpy(dtype=float)

coefficients = np.linalg.lstsq(x_train, y_train, rcond=None)[0]
coefficients_df = pd.DataFrame({
    'feature_name': ['intercept'] + feature_cols,
    'coefficient': coefficients,
})
coefficients_df


,feature_name,coefficient
0,intercept,28.105848
1,time_index,1.550829
2,month_sin,-13.644660
3,month_cos,2.762063
4,lag_1,0.470852
5,lag_2,-0.145292
6,lag_3,0.060043
7,rolling_mean_3,0.128534


In [19]:
x_valid = validation_df[feature_cols].to_numpy(dtype=float)
x_valid = np.column_stack([np.ones(len(x_valid)), x_valid])

validation_df = validation_df.copy()
validation_df['predicted_value'] = x_valid @ coefficients
validation_df['absolute_error'] = (validation_df['selected_metric'] - validation_df['predicted_value']).abs()

validation_df[['month_start', 'selected_metric', 'predicted_value', 'absolute_error']]


,month_start,selected_metric,predicted_value,absolute_error
43,2008-12-01,227.0,212.122245,14.877755
44,2009-01-01,219.0,213.069331,5.930669
45,2009-02-01,197.0,205.120877,8.120877
46,2009-03-01,192.0,192.569349,0.569349
47,2009-04-01,153.0,193.429455,40.429455
48,2009-05-01,128.0,177.178149,49.178149


In [20]:
validation_mae = float(validation_df['absolute_error'].mean())
validation_mae


19.851042374608912

In [21]:
working_df = model_history_df.copy().sort_values('month_start').reset_index(drop=True)
forecast_rows = []

for step in range(1, 4):
    next_month = working_df['month_start'].max() + pd.offsets.MonthBegin(1)
    candidate_df = pd.concat(
        [working_df, pd.DataFrame([{'month_start': next_month, 'selected_metric': np.nan}])],
        ignore_index=True,
    )
    candidate_features_df = add_time_series_features(candidate_df)
    next_row = candidate_features_df.iloc[[-1]].copy()
    x_next = next_row[feature_cols].to_numpy(dtype=float)
    x_next = np.column_stack([np.ones(len(x_next)), x_next])
    predicted_value = max(0.0, float((x_next @ coefficients)[0]))
    forecast_rows.append({
        'month_start': next_month,
        'predicted_value': predicted_value,
        'predicted_value_rounded': int(round(predicted_value)),
        'forecast_step': step,
    })
    working_df = pd.concat(
        [working_df, pd.DataFrame([{'month_start': next_month, 'selected_metric': predicted_value}])],
        ignore_index=True,
    )

forecast_df = pd.DataFrame(forecast_rows)
forecast_df


,month_start,predicted_value,predicted_value_rounded,forecast_step
0,2009-06-01,175.819880,176,1
1,2009-07-01,207.676572,208,2
2,2009-08-01,224.126119,224,3


In [22]:
for _, row in forecast_df.iterrows():
    print(f"{row['month_start'].date()}: {row['predicted_value']:.2f} (rounded={int(row['predicted_value_rounded'])})")


2009-06-01: 175.82 (rounded=176)
2009-07-01: 207.68 (rounded=208)
2009-08-01: 224.13 (rounded=224)


When the logic is clear, move to `05_exercise_3_production_solution.ipynb` to generate the final forecast files quickly.


In [ ]:
spark.stop()
